In [1]:
import pandas as pd
from datetime import datetime
import os

# ==========================================
# 1. Configuration and Paths
# ==========================================

# Base Paths (Uncomment the active path)
# BASE_PATH = 'C:\\Users\\Edmilson\\Gama_Workspace2\\ABMS-WP'
# BASE_PATH = 'C:\\Users\\e124796\\Gama_Workspace\\ABMS-WP'
BASE_PATH = 'E:\\Projetos\\ABMS-WP' # Currently active path

# Files
INPUT_FILE = 'Tabela_consumo_Itapua_120m.csv'
OUTPUT_FILE = 'Tabela_consumo_Itapua_120m_por_mes.csv'

def main():
    print("--- Starting Monthly Consumption Aggregation (120 Months) ---")

    # ==========================================
    # 2. Load Data
    # ==========================================
    input_path = os.path.join(BASE_PATH, 'includes', INPUT_FILE)
    print(f"Reading file: {input_path}")
    
    # Reading CSV with semicolon delimiter
    df = pd.read_csv(input_path, sep=';', quotechar='"')

    # ==========================================
    # 3. Data Processing
    # ==========================================
    
    # Convert reading date to datetime object
    df['DT_LEITURA'] = pd.to_datetime(df['DT_LEITURA'])

    # Convert consumption units (e.g., Liters to m3) if needed
    # df['HCLQTCON'] = df['HCLQTCON'] / 1000

    # Aggregate by Reference Month/Year (AM_REFERENCIA)
    # Summing volume (HCLQTCON) and taking the first date
    result_df = df.groupby('AM_REFERENCIA').agg({
        'HCLQTCON': 'sum',
        'DT_LEITURA': 'first' 
    }).reset_index()

    # Rename columns to target format
    result_df = result_df.rename(columns={
        'AM_REFERENCIA': 'mes/ano',
        'HCLQTCON': 'consumo',
        'DT_LEITURA': 'data'
    })

    # Format 'data' column to be the first day of the month
    # Logic: Convert YYYYMM to YYYYMM01 then to Date object
    result_df['data'] = result_df['mes/ano'].apply(
        lambda x: datetime.strptime(str(x) + '01', '%Y%m%d').date()
    )

    # Sort by chronological order
    result_df = result_df.sort_values('mes/ano')

    # Reorder columns
    result_df = result_df[['mes/ano', 'consumo', 'data']]

    # ==========================================
    # 4. Save Results
    # ==========================================
    output_path = os.path.join(BASE_PATH, 'includes', OUTPUT_FILE)
    
    # Saving with semicolon separator
    result_df.to_csv(output_path, index=False, sep=';', quotechar='"')

    print(f"File successfully generated: {output_path}")
    print("\nSample Data:")
    print(result_df.head())

if __name__ == "__main__":
    main()

--- Starting Monthly Consumption Aggregation (120 Months) ---
Reading file: E:\Projetos\ABMS-WP\includes\Tabela_consumo_Itapua_120m.csv
File successfully generated: E:\Projetos\ABMS-WP\includes\Tabela_consumo_Itapua_120m_por_mes.csv

Sample Data:
   mes/ano  consumo        data
0   201501   202594  2015-01-01
1   201502   197816  2015-02-01
2   201503   202908  2015-03-01
3   201504   185771  2015-04-01
4   201505   189212  2015-05-01
